# Actividad final: Minimax

## Taller de Tres en raya

Este notebook desarrolla el taller de búsqueda adversarial con el algoritmo **Minimax**. Se modelan los estados del juego, las acciones legales, la transición entre tableros, los estados terminales y la utilidad desde la perspectiva de `X` (MAX).

La implementación supone que `X` inicia la partida y que los jugadores alternan turnos. En un estado no terminal, `es_max=True` representa el turno de `X` y `es_max=False` representa el turno de `O`.

In [1]:
VACIO = " "
JUGADORES = ("X", "O")
JUGADAS_GANADORAS = (
    (0, 1, 2), (3, 4, 5), (6, 7, 8),
    (0, 3, 6), (1, 4, 7), (2, 5, 8),
    (0, 4, 8), (2, 4, 6),
)


def acciones(tablero):
    """Devuelve los índices de las casillas vacías."""
    return tuple(indice for indice, casilla in enumerate(tablero) if casilla == VACIO)


def resultado(tablero, accion, jugador):
    """Devuelve un tablero nuevo con la jugada aplicada."""
    if jugador not in JUGADORES:
        raise ValueError("El jugador debe ser 'X' u 'O'.")
    if accion not in range(9):
        raise ValueError("La acción debe ser un índice entre 0 y 8.")
    if tablero[accion] != VACIO:
        raise ValueError("La casilla seleccionada no está vacía.")

    nuevo_tablero = list(tablero)
    nuevo_tablero[accion] = jugador
    return tuple(nuevo_tablero)


def hay_ganador(tablero, jugador):
    return any(all(tablero[posicion] == jugador for posicion in linea)
               for linea in JUGADAS_GANADORAS)


def terminal(tablero):
    """Indica si hay victoria o si el tablero está lleno."""
    return (
        hay_ganador(tablero, "X")
        or hay_ganador(tablero, "O")
        or not acciones(tablero)
    )


def utilidad(tablero):
    """Evalúa un estado terminal desde la perspectiva de MAX (X)."""
    if hay_ganador(tablero, "X"):
        return 1
    if hay_ganador(tablero, "O"):
        return -1
    if not acciones(tablero):
        return 0
    raise ValueError("La utilidad solo está definida para estados terminales.")


def minimax_tictactoe(tablero, es_max):
    """Calcula el valor óptimo del estado mediante Minimax."""
    if terminal(tablero):
        return utilidad(tablero)

    jugador = "X" if es_max else "O"
    valores = [
        minimax_tictactoe(resultado(tablero, accion, jugador), not es_max)
        for accion in acciones(tablero)
    ]
    return max(valores) if es_max else min(valores)


def mejor_jugada_tictactoe(tablero, es_max=True):
    """Devuelve la mejor acción y su valor para el jugador del turno."""
    if terminal(tablero):
        raise ValueError("No hay jugadas en un estado terminal.")

    jugador = "X" if es_max else "O"
    opciones = [
        (
            minimax_tictactoe(resultado(tablero, accion, jugador), not es_max),
            accion,
        )
        for accion in acciones(tablero)
    ]
    return (max if es_max else min)(opciones, key=lambda opcion: opcion[0])


TABLERO_VACIO = (VACIO,) * 9
TABLERO_VICTORIA_X = ("X", "X", "X", "O", "O", VACIO, VACIO, VACIO, VACIO)
TABLERO_VICTORIA_O = ("O", "O", "O", "X", "X", VACIO, VACIO, VACIO, VACIO)

assert acciones(TABLERO_VACIO) == tuple(range(9))
assert resultado(TABLERO_VACIO, 4, "X")[4] == "X"
assert terminal(TABLERO_VICTORIA_X)
assert utilidad(TABLERO_VICTORIA_X) == 1
assert utilidad(TABLERO_VICTORIA_O) == -1
assert minimax_tictactoe(TABLERO_VACIO, True) == 0

print("Pruebas básicas superadas")
print("Valor del tablero vacío:", minimax_tictactoe(TABLERO_VACIO, True))
print("Mejor jugada inicial de X:", mejor_jugada_tictactoe(TABLERO_VACIO))

Pruebas básicas superadas
Valor del tablero vacío: 0
Mejor jugada inicial de X: (0, 0)


In [2]:
print("=== Resultados de las funciones principales ===")
print("Acciones disponibles en el tablero vacío:", acciones(TABLERO_VACIO))

resultado_ejemplo = resultado(TABLERO_VACIO, 4, "X")
print("Resultado de colocar X en la casilla 4:", resultado_ejemplo)
print("¿El tablero de victoria de X es terminal?:", terminal(TABLERO_VICTORIA_X))
print("Utilidad del tablero de victoria de X:", utilidad(TABLERO_VICTORIA_X))
print("Utilidad del tablero de victoria de O:", utilidad(TABLERO_VICTORIA_O))
print("Valor Minimax del tablero vacío:", minimax_tictactoe(TABLERO_VACIO, True))
print("Mejor jugada inicial para MAX:", mejor_jugada_tictactoe(TABLERO_VACIO))

=== Resultados de las funciones principales ===
Acciones disponibles en el tablero vacío: (0, 1, 2, 3, 4, 5, 6, 7, 8)
Resultado de colocar X en la casilla 4: (' ', ' ', ' ', ' ', 'X', ' ', ' ', ' ', ' ')
¿El tablero de victoria de X es terminal?: True
Utilidad del tablero de victoria de X: 1
Utilidad del tablero de victoria de O: -1
Valor Minimax del tablero vacío: 0
Mejor jugada inicial para MAX: (0, 0)


## 1. Modelo del problema

- **Estado:** tupla de nueve posiciones.
- **Acción:** índice de una casilla vacía, entre `0` y `8`.
- **Resultado:** copia del tablero con la marca del jugador en la casilla elegida.
- **Estado terminal:** existe una línea ganadora o no quedan casillas vacías.
- **Utilidad:** `+1` si gana `X`, `-1` si gana `O` y `0` si hay empate.

Minimax explora todas las continuaciones posibles. MAX selecciona el mayor valor y MIN selecciona el menor valor, suponiendo que ambos jugadores actúan racionalmente.

In [3]:
# Árbol de tres niveles: MAX -> MIN -> MAX -> utilidad.
# En cada nivel, Minimax alterna el jugador que debe elegir.
arbol_tres_niveles = {
    "A": ["B", "C"],
    "B": ["D", "E"],
    "C": ["F", "G"],
}
utilidades_tres_niveles = {
    "D": 3,
    "E": 5,
    "F": 9,
    "G": 1,
}


def minimax_arbol(nodo, es_max, arbol, utilidades):
    if nodo in utilidades:
        return utilidades[nodo]

    valores = [
        minimax_arbol(hijo, not es_max, arbol, utilidades)
        for hijo in arbol[nodo]
    ]
    return max(valores) if es_max else min(valores)


valor_b = min(utilidades_tres_niveles["D"], utilidades_tres_niveles["E"])
valor_c = min(utilidades_tres_niveles["F"], utilidades_tres_niveles["G"])
valor_a_manual = max(valor_b, valor_c)
valor_a_python = minimax_arbol("A", True, arbol_tres_niveles, utilidades_tres_niveles)

assert (valor_b, valor_c, valor_a_manual) == (3, 1, 3)
assert valor_a_python == valor_a_manual
print("Valores manuales: B =", valor_b, ", C =", valor_c, ", A =", valor_a_manual)
print("Valor calculado por Python:", valor_a_python)
print("Jugada elegida por MAX: B")

Valores manuales: B = 3 , C = 1 , A = 3
Valor calculado por Python: 3
Jugada elegida por MAX: B


### Representación gráfica del árbol Minimax

![Árbol de decisión Minimax](images/image.png)

### Cálculo manual del árbol

El nodo `B` pertenece a MIN, por lo que su valor es:

$$B = \min(3, 5) = 3$$

El nodo `C` también pertenece a MIN:

$$C = \min(9, 1) = 1$$

Finalmente, `A` pertenece a MAX:

$$A = \max(B, C) = \max(3, 1) = 3$$

Por tanto, MAX elige la rama `B`. La celda siguiente comprueba este cálculo con una implementación recursiva.

In [4]:
tableros_prueba = {
    "X gana en la fila superior": TABLERO_VICTORIA_X,
    "O gana en la fila superior": TABLERO_VICTORIA_O,
    "Empate": (
        "X", "O", "X",
        "X", "O", "O",
        "O", "X", "X",
    ),
    "X debe bloquear a O": (
        "O", "O", VACIO,
        "X", VACIO, VACIO,
        VACIO, VACIO, "X",
    ),
}

for nombre, tablero in tableros_prueba.items():
    if terminal(tablero):
        valor = utilidad(tablero)
    else:
        valor = minimax_tictactoe(tablero, True)
    print(f"{nombre}: terminal={terminal(tablero)}, valor={valor}")

bloqueo = tableros_prueba["X debe bloquear a O"]
valor_bloqueo, accion_bloqueo = mejor_jugada_tictactoe(bloqueo)
assert accion_bloqueo == 2
print(f"En el tablero de bloqueo, MAX juega la casilla {accion_bloqueo} con valor {valor_bloqueo}.")

X gana en la fila superior: terminal=True, valor=1
O gana en la fila superior: terminal=True, valor=-1
Empate: terminal=True, valor=0
X debe bloquear a O: terminal=False, valor=1
En el tablero de bloqueo, MAX juega la casilla 2 con valor 1.


## 2. Modificación del juego de las piedras

Se modifica el juego original para permitir retirar únicamente `1`, `2` o `4` piedras. Gana el jugador que retira la última piedra. La utilidad continúa expresándose desde la perspectiva de MAX: `+1` si gana MAX y `-1` si gana MIN.

In [5]:
MOVIMIENTOS_PIEDRAS = (1, 2, 4)


def movimientos_validos_piedras(piedras):
    return tuple(movimiento for movimiento in MOVIMIENTOS_PIEDRAS if movimiento <= piedras)


def minimax_piedras_modificado(piedras, es_max):
    if piedras == 0:
        return -1 if es_max else 1

    valores = [
        minimax_piedras_modificado(piedras - movimiento, not es_max)
        for movimiento in movimientos_validos_piedras(piedras)
    ]
    return max(valores) if es_max else min(valores)


def mejor_movimiento_piedras_modificado(piedras):
    opciones = [
        (
            minimax_piedras_modificado(piedras - movimiento, False),
            movimiento,
        )
        for movimiento in movimientos_validos_piedras(piedras)
    ]
    return max(opciones, key=lambda opcion: opcion[0])


resultados_piedras = []
for piedras in range(1, 13):
    valor = minimax_piedras_modificado(piedras, True)
    valor_mejor_movimiento, mejor_movimiento = mejor_movimiento_piedras_modificado(piedras)
    resultados_piedras.append((piedras, valor, mejor_movimiento))
    print(
        f"{piedras:2d} piedras -> valor={valor:+d}, "
        f"mejor movimiento={mejor_movimiento}"
    )

assert resultados_piedras[0] == (1, 1, 1)
assert resultados_piedras[2][1] == -1
assert resultados_piedras[3][1] == 1

 1 piedras -> valor=+1, mejor movimiento=1
 2 piedras -> valor=+1, mejor movimiento=2
 3 piedras -> valor=-1, mejor movimiento=1
 4 piedras -> valor=+1, mejor movimiento=1
 5 piedras -> valor=+1, mejor movimiento=2
 6 piedras -> valor=-1, mejor movimiento=1
 7 piedras -> valor=+1, mejor movimiento=1
 8 piedras -> valor=+1, mejor movimiento=2
 9 piedras -> valor=-1, mejor movimiento=1
10 piedras -> valor=+1, mejor movimiento=1
11 piedras -> valor=+1, mejor movimiento=2
12 piedras -> valor=-1, mejor movimiento=1


### Análisis de resultados

1. El tablero vacío tiene valor `0` para `X` bajo Minimax, lo que indica que, con juego perfecto, el estado inicial de tres en raya es neutral: ni `X` ni `O` tienen una ventaja garantizada.
2. Las pruebas de terminalidad muestran que el algoritmo reconoce correctamente los casos finales: una línea de `X` devuelve utilidad `+1`, una línea de `O` devuelve `-1` y un empate devuelve `0`.
3. En situaciones con amenaza inmediata, Minimax no solo considera la jugada más agresiva, sino la respuesta del adversario, por ejemplo, en el caso “X debe bloquear a O”, la mejor acción es la casilla `2`, porque bloquea la victoria del rival y maximiza el valor del estado.
4. Esto valida la idea central del algoritmo: cada movimiento se evalúa según el peor resultado posible que puede producir, y se elige la alternativa cuyo mínimo valor sea el mejor entre todas las opciones.
5. En el juego de piedras, las cantidades `3, 6, 9, 12, ...` son posiciones perdedoras para MAX. En ellas, cualquier movimiento deja al oponente una cantidad que no es múltiplo de `3`.
6. El patrón se repite cada tres piedras porque los movimientos `1` y `2` permiten responder y completar grupos de tres; el movimiento `4` es equivalente a retirar `1` respecto al patrón modular.
7. Una posición puede ser perdedora aunque queden varias piedras porque Minimax considera todas las respuestas del oponente, no solamente la jugada inmediata.
8. Puede haber varias jugadas con el mismo valor óptimo. La función conserva la primera según el orden definido en `MOVIMIENTOS_PIEDRAS`.
9. Si solo se permitieran `1` o `2` piedras, el patrón de posiciones perdedoras cambiaría, pero el razonamiento por estados y respuestas sucesivas sería el mismo.

### Pregunta final

Una decisión aparentemente buena puede ser mala después de considerar al adversario porque el oponente puede responder de forma que elimine la ventaja inicial. Minimax evita evaluar únicamente el beneficio inmediato y escoge la acción cuyo peor resultado posible sea el mejor entre las alternativas.

## Uso de IA generativa

- **Herramienta utilizada:** GitHub Copilot.
- **Propósito de uso:** apoyo para revisar la estructura del taller, sugerir casos de prueba y detectar inconsistencias durante la ejecución del notebook.
- **Partes en las que fue empleada:** organización inicial de las celdas, revisión de pruebas y apoyo en la redacción de algunas explicaciones.
